# PayRevive: data and feature audit

**Question:** what information do our generated payment observations contain, and what can we legitimately use at diagnosis time?

This notebook uses actual checksummed simulator artifacts. It imports the same loading, validation, reporting and plotting functions as the audit CLI; it implements no separate business or ML logic.

Before an interactive run, generate `work/datasets/demo` from the repository root with `uv run python -m simulator.run --output work/datasets/demo`. Then launch this notebook using the locked analysis environment. The headless runner supplies explicit paths and verifies its Python kernel.

The seed-42 corpus is exploratory. A withheld month here is a boundary demonstration, **not a claim of an untouched final model test set**. No detector, classifier, feature-selection result or SHAP explanation exists in this notebook.


In [ ]:
import os
from pathlib import Path
from uuid import uuid4

import pandas as pd
from IPython.display import Image, display

from ml.analysis import audit as audit_module
from ml.analysis.audit import run_audit

PROJECT_ROOT = Path(audit_module.__file__).resolve().parents[2]
DATASET = Path(os.environ.get("PAYREVIVE_NOTEBOOK_DATASET", PROJECT_ROOT / "work/datasets/demo"))
ARTIFACTS = Path(
    os.environ.get(
        "PAYREVIVE_NOTEBOOK_OUTPUT",
        PROJECT_ROOT / "work/analysis" / f"interactive-{uuid4().hex[:8]}",
    )
)
report = run_audit(
    DATASET,
    ARTIFACTS,
    until=os.environ.get("PAYREVIVE_NOTEBOOK_UNTIL") or None,
    expected_manifest_sha256=os.environ.get("PAYREVIVE_EXPECTED_MANIFEST"),
)

## 1. What was actually loaded?

The loader checks file hashes, producer/configuration identity, organization ownership, monthly cohorts, entity joins, UTC observation times, and prior-history values. Operational loading does not read hidden truth. The audit explicitly opens evaluation truth separately, then limits displayed statistics to the exploration period.

The default exploration ends at the start of the final generated month. Counts for withheld observations are shown without their outcome breakdown. Future model selection must freeze an appropriate evaluation protocol and reserve new data.


In [ ]:
display(pd.DataFrame([report["provenance"]]).T.rename(columns={0: "value"}))
display(pd.DataFrame([report["population"]]).T.rename(columns={0: "value"}))
display(pd.DataFrame([report["monthly_explored_counts"]]))

## 2. What does the population look like?

These distributions come from synthetic mechanisms: segment and subscription amount are coupled; method expiry and transient degradation influence failure hazards. The associations are useful engineering fixtures, not evidence about real customers or real banks.

The failure-category panel uses hidden labels for analysis only. A classifier must never receive those labels, disclosed reasons, IDs, or incident flags as features. All initial outcomes are retained for EDA; later diagnosis training will operate on failed observations. Detection needs successful attempts in its denominator.


In [ ]:
display(Image(filename=str(ARTIFACTS / "overview.png")))
display(pd.DataFrame(report["class_distribution"]))

## 3. Which candidate features are available at the right time?

The table is the shared contract's rationale, not a list of proven useful features. We will compare candidate subsets using validation data later.

Latency is known after the response: it can help diagnose an observed failure, but cannot support a pre-attempt prediction. History excludes the current payment and future observations. Tenure is baseline population metadata in generator v1. Day/hour refer to UTC observation time.

The generator's expiry association is probabilistic; it does not reproduce processor enforcement. We must not infer permission to retry from those synthetic correlations.


In [ ]:
display(pd.DataFrame(report["feature_catalog"]))
display(pd.DataFrame(report["feature_statistics"]))
display(pd.DataFrame({"excluded_join_metadata": report["excluded_join_metadata"]}))

## 4. Are the deliberately injected incidents visible?

The following curves use the same explored observations. Shading marks known injected windows for each scope; it does not mean every segment was affected. Empty days remain gaps. No alert threshold was tuned and no detector precision or recall is claimed.

This view motivates the next milestone: causal-in-time rolling baselines/CUSUM, minimum traffic requirements, threshold selection on validation periods, and held-out incident/false-alert evaluation.


In [ ]:
display(Image(filename=str(ARTIFACTS / "temporal.png")))

## 5. What can we conclude, and what remains unproven?

- We can verify the feature contracts and observe the simulator's designed relationships.
- Missing reported reasons and imbalanced labels motivate baseline comparisons; they do not justify a particular model yet.
- IDs remain join metadata. Disclosed reason is reserved for deterministic mapping, not predicting itself.
- Preprocessing, feature selection, model choice and threshold tuning must use training/validation data only.
- Repeated customers across time answer a different question from unseen-customer generalization. Changed simulator scenarios are also needed; a new seed alone is not a new data-generating process.
- SHAP will explain model contributions after a real model is trained, not establish causal explanations.
- Monetary policy experiments and uncertainty intervals remain a separate later milestone.

For interviews, explain why each input is available, how leakage is prevented, and which conclusions are limited to the simulator. The environment/source record below binds this result to actual code and dependencies.


In [ ]:
print("Python:", report["environment"]["python_executable"])
display(pd.DataFrame([report["environment"]["dependencies"]]).T.rename(columns={0: "version"}))
print("Synthetic only:", report["synthetic_only"])
print("Final test claim:", report["final_test_claim"])